# Детектор фишинговых писем

**Задача:** классифицировать электронные письма как безопасные или фишинговые и сравнить два классических baseline-подхода.

**Стек:** Hugging Face Datasets, scikit-learn, pandas, seaborn.

Датасет содержит шаблонный вопрос и ответ с меткой прямо внутри текста. Эти фрагменты удаляются до обучения, иначе возникает прямая утечка целевого признака.


In [ ]:
import re

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    classification_report,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline

SEED = 42


In [ ]:
dataset = load_dataset("drorrabin/phishing_emails-data", split="train")
data = dataset.to_pandas()

text_column = next(
    (name for name in ["Email Text", "text", "email"] if name in data.columns),
    None,
)
label_column = next(
    (name for name in ["label", "Email Type", "email_type", "Class"] if name in data.columns),
    None,
)
if text_column is None or label_column is None:
    raise ValueError(f"Неожиданная схема датасета: {data.columns.tolist()}")


def remove_label_leakage(text: str) -> str:
    text = str(text)
    text = re.sub(
        r"^\s*Is the following email safe or phishing\?+\s*",
        "",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(
        r"Email type is:\s*(?:phishing|safe)\s*email\s*$",
        "",
        text,
        flags=re.IGNORECASE,
    )
    return re.sub(r"\s+", " ", text).strip()


def normalize_label(value) -> int:
    if isinstance(value, str):
        return int("phishing" in value.lower())
    return int(value)


data = data[[text_column, label_column]].dropna().copy()
data["text"] = data[text_column].map(remove_label_leakage)
data["label"] = data[label_column].map(normalize_label)
data = data[data["text"].str.len() > 0].drop_duplicates("text").reset_index(drop=True)

print(data["label"].value_counts().rename(index={0: "safe", 1: "phishing"}))


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data["text"],
    data["label"],
    test_size=0.25,
    random_state=SEED,
    stratify=data["label"],
)


def make_pipeline(classifier) -> Pipeline:
    return Pipeline(
        [
            (
                "tfidf",
                TfidfVectorizer(
                    stop_words="english",
                    ngram_range=(1, 2),
                    min_df=2,
                    max_features=30_000,
                    sublinear_tf=True,
                ),
            ),
            ("classifier", classifier),
        ]
    )


models = {
    "Logistic Regression": make_pipeline(
        LogisticRegression(max_iter=2_000, class_weight="balanced", random_state=SEED)
    ),
    "Complement Naive Bayes": make_pipeline(ComplementNB(alpha=0.5)),
}


In [ ]:
rows = []
stored_predictions = {}
stored_probabilities = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)
    probability = model.predict_proba(X_test)[:, 1]
    stored_predictions[name] = prediction
    stored_probabilities[name] = probability
    rows.append(
        {
            "model": name,
            "precision": precision_score(y_test, prediction),
            "recall": recall_score(y_test, prediction),
            "f1": f1_score(y_test, prediction),
            "roc_auc": roc_auc_score(y_test, probability),
            "pr_auc": average_precision_score(y_test, probability),
            "mcc": matthews_corrcoef(y_test, prediction),
        }
    )

metrics = pd.DataFrame(rows).set_index("model").sort_values("f1", ascending=False)
metrics.round(4)


In [ ]:
best_model_name = metrics.index[0]
best_model = models[best_model_name]
best_prediction = stored_predictions[best_model_name]
best_probability = stored_probabilities[best_model_name]

print(classification_report(y_test, best_prediction, target_names=["safe", "phishing"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    best_prediction,
    display_labels=["safe", "phishing"],
    cmap="Blues",
    ax=axes[0],
)
axes[0].set_title(best_model_name)
RocCurveDisplay.from_predictions(y_test, best_probability, ax=axes[1])
axes[1].set_title("ROC-кривая")
plt.tight_layout()


In [ ]:
sample = X_test.sample(n=min(10, len(X_test)), random_state=SEED)
sample_probability = best_model.predict_proba(sample)[:, 1]
pd.DataFrame(
    {
        "email": sample.values,
        "true_label": y_test.loc[sample.index].values,
        "phishing_probability": sample_probability,
    }
).sort_values("phishing_probability", ascending=False)


## Практические замечания

В рабочей системе порог классификации следует выбирать по стоимости ошибок: пропущенный фишинг обычно дороже ложной тревоги. Дополнительно нужны временное разбиение, дедупликация цепочек писем и проверка на данных из другого источника.
